# Criterios, verificación y una persona que aprueba

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un agente al que se le pide «identificá los riesgos» entrega un informe prolijo que nadie puede auditar: no se sabe con qué criterio decidió. Con criterios explícitos se puede comparar su resultado con una regla, y con una compuerta humana se decide qué se guarda. |
| **Producto** | Una comparación entre un pedido ambiguo y uno con criterios, una verificación automática contra la regla y una compuerta que guarda el informe solo si una persona lo aprueba y los datos coinciden. |
| **Recursos** | Ollama con `granite4:latest`. La ejecución completa lleva unos cinco minutos. Gemini es opcional (seis corridas). |

La cooperativa quiere que el agente revise las parcelas y avise cuáles necesitan atención. Un técnico de campo va a leer el informe y, según lo que diga, va a salir o no a una parcela. Si el agente decide con un criterio que nadie conoce, el técnico no puede saber si confiar. Y si el informe se guarda solo, nadie responde por él.

**¿Cómo hacemos que el trabajo de un agente se pueda verificar, y en qué punto tiene que intervenir una persona?**

> **Cómo se trabaja.** En clase se presenta cada tema con el código ya escrito. Después lo explorás por tu cuenta: ejecutás las celdas, leés las notas **Para mirar** y cambiás valores para ver qué pasa. Al final de cada bloque hay un **microlaboratorio**: una tabla te dice qué líneas completar y con qué, y la celda siguiente comprueba si quedó bien.

## Los datos

| Campo | Dato |
|---|---|
| Fuente | `datos/parcelas.csv`: ocho parcelas, **datos sintéticos** escritos para el curso (ver `datos/LEEME.md`). |
| Regla de referencia | La misma del cuaderno 02: alerta si la temperatura es de 34 °C o más, la humedad de 40 % o menos, la lluvia es 0 mm o el estado del cultivo no es `normal`. Es una regla didáctica, no un criterio agronómico. |
| Limitaciones | Ocho filas, una regla simple, un modelo chico. Cada variante se corre tres veces: muestra tendencias, no probabilidades. El cultivo real exige criterio técnico que este cuaderno no reemplaza. |

In [1]:
import utils  # va primero: ordena los avisos de las librerias que siguen
import time
from pathlib import Path

from smolagents import CodeAgent, tool

utils.comprobar_modelo_local()
utils.versiones(["smolagents", "ollama"])

Ollama responde en http://localhost:11434 y tiene granite4:latest.


{'python': '3.12.13', 'smolagents': '1.26.0', 'ollama': '0.6.3'}

In [2]:
@tool
def leer_parcelas() -> list:
    """Devuelve las mediciones de las parcelas como una lista de diccionarios.

    Cada diccionario tiene exactamente estas claves: "parcela" (letra),
    "temperatura_c" (numero), "humedad_pct" (numero), "lluvia_mm" (numero)
    y "estado_cultivo" (texto).

    Returns:
        Lista con un diccionario por parcela.
    """
    return utils.cargar_parcelas()


def calcular_alertas(parcelas_a_revisar: list[dict]) -> list[str]:
    """Aplica la regla de alerta a cada parcela.

    Args:
        parcelas_a_revisar: Lista de diccionarios con las mediciones.

    Returns:
        Las letras de las parcelas que cumplen una o mas condiciones.
    """
    en_alerta = []
    for parcela in parcelas_a_revisar:
        if parcela["temperatura_c"] >= 34:
            en_alerta.append(parcela["parcela"])
        elif parcela["humedad_pct"] <= 40:
            en_alerta.append(parcela["parcela"])
        elif parcela["lluvia_mm"] == 0:
            en_alerta.append(parcela["parcela"])
        elif parcela["estado_cultivo"] != "normal":
            en_alerta.append(parcela["parcela"])
    return en_alerta


alertas_correctas = calcular_alertas(utils.cargar_parcelas())
print("Parcelas en alerta según la regla:", alertas_correctas)

Parcelas en alerta según la regla: ['B', 'D', 'E', 'F', 'G']


## 1. El mismo agente, dos pedidos

Pedimos lo mismo de dos maneras. Los dos reciben la misma herramienta y deben devolver un diccionario con las parcelas y un informe breve:

- **Ambiguo**: «identificá cuáles tienen mayor nivel de riesgo». No dice qué es riesgo.
- **Con criterios**: incluye la regla de alerta.

Cada pedido se corre tres veces, con tres semillas distintas (la semilla fija el azar de cada corrida, pero el detalle puede variar entre computadoras y versiones: lo que se mantiene es la tendencia).

In [3]:
formato_de_salida = (
    "Devolvé con final_answer un diccionario con dos claves: 'parcelas' "
    "(una lista con las letras de las parcelas) y 'informe' (dos o tres "
    "oraciones que justifiquen la lista)."
)
reglas = (
    "Una parcela está en alerta si cumple una o más de estas condiciones: "
    "temperatura_c >= 34; humedad_pct <= 40; lluvia_mm == 0; "
    "estado_cultivo distinto de 'normal'."
)

pedidos = {
    "ambiguo": (
        "Usá la herramienta leer_parcelas. Analizá los datos e identificá "
        f"cuáles parcelas tienen mayor nivel de riesgo. {formato_de_salida}"
    ),
    "con criterios": (
        f"Usá la herramienta leer_parcelas. {reglas} Indicá cuáles "
        f"parcelas están en alerta. {formato_de_salida}"
    ),
}


def parcelas_de(respuesta: object) -> list[str]:
    """Extrae la lista de parcelas de la respuesta de un agente.

    Args:
        respuesta: Lo que devolvio el agente con final_answer.

    Returns:
        Las letras ordenadas, o una lista vacia si la respuesta no es un
        diccionario con la clave "parcelas".
    """
    if not isinstance(respuesta, dict):
        return []
    lista = respuesta.get("parcelas", [])
    if not isinstance(lista, list):
        return []
    letras = []
    for letra in lista:
        letras.append(str(letra).strip().upper())
    return sorted(letras)

In [4]:
resultados = {}
filas_de_pedidos = []
for nombre_del_pedido, texto_del_pedido in pedidos.items():
    resultados[nombre_del_pedido] = []
    for semilla in (1, 2, 3):
        modelo_con_semilla = utils.crear_modelo_local(
            temperatura=0.7, semilla=semilla
        )
        agente = CodeAgent(
            tools=[leer_parcelas], model=modelo_con_semilla,
            max_steps=5, verbosity_level=-1,
        )
        respuesta = agente.run(texto_del_pedido)
        resultados[nombre_del_pedido].append(respuesta)
        filas_de_pedidos.append({
            "pedido": nombre_del_pedido,
            "semilla": semilla,
            "parcelas que devolvió": parcelas_de(respuesta),
            "tipo": type(respuesta).__name__,
        })

utils.tabla(filas_de_pedidos)

,pedido,semilla,parcelas que devolvió,tipo
0,ambiguo,1,[],dict
1,ambiguo,2,[],AgentText
2,ambiguo,3,[],AgentText
3,con criterios,1,[],AgentText
4,con criterios,2,"[B, D, E, F, G]",dict
5,con criterios,3,"[B, D, E, F, G]",dict


In [5]:
numero = 0
for respuesta in resultados["ambiguo"]:
    numero = numero + 1
    print(f"Pedido ambiguo, corrida {numero}:")
    print(str(respuesta)[:220].replace("\n", " "))
    print()

Pedido ambiguo, corrida 1:
{'parcelas': [], 'informe': 'Las parcelas con mayor nivel de riesgo son aquellas que tienen temperaturas superiores a 30 grados Celsius y humedad porcentual menor al 20%, lo cual indica un ambiente propicio para el creci

Pedido ambiguo, corrida 2:
<code> import json  risk_parcelas = [] for parcela in parcelas:     if parcela['estado_cultivo'] == "hojas marchitas" or parcela['estado_cultivo'] == "crecimiento lento" or parcela['estado_cultivo'] == "hojas amarillenta

Pedido ambiguo, corrida 3:
<code> parcelas = leer_parcelas() risk_parcelas = [] for parcela in parcelas:     if parcela['temperatura_c'] > 30 and (parcela['humedad_pct'] < 40 or parcela['lluvia_mm'] == 0):         risk_parcelas.append(parcela['par



> **Para mirar.**
>
> - El pedido ambiguo no produjo en ninguna corrida una lista de parcelas usable: dos respondieron con código sin ejecutar y la otra devolvió un diccionario con la lista vacía.
> - Sin una regla, el modelo inventó una: el informe de la corrida 1 habla de «temperaturas superiores a 30 grados» y «humedad menor al 20 %», umbrales que no figuran en ninguna parte, y el código de la corrida 2 filtra por estados como `hojas marchitas`, que no existen en los datos.
> - Con criterios, dos de las tres corridas devolvieron `B, D, E, F, G` y la tercera respondió texto. Tres corridas por pedido muestran una tendencia, no una probabilidad.

### Microlaboratorio 1 — Comparar con la regla

**Tarea.** Escribir la función que dice en qué se diferencia el resultado del agente de la regla.  
**Producto.** `comparar_con_la_regla`, comprobada con listas inventadas.  
**Antes de empezar.** Sección 1.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) condición del primer `if` | verdadera cuando una letra de la regla **no** está en la lista del agente | `letra not in del_agente` |
| (2) condición del segundo `if` | verdadera cuando una letra del agente **no** está en la lista de la regla | `letra not in correctas` |

**Comprobación.** `["A", "B"]` contra `["B", "C"]` tiene que dar `faltan = ["C"]` y `sobran = ["A"]`, y dos listas iguales tienen que dar `coincide = True`.

> **Ayuda 1.** `faltan` son las que la regla marca y el agente no; `sobran`, al revés.  
> **Ayuda 2.** `letra not in lista` es verdadero cuando la letra no aparece en la lista.

In [6]:
def comparar_con_la_regla(del_agente: list[str], correctas: list[str]) -> dict:
    """Compara las parcelas que dijo el agente con las de la regla.

    Args:
        del_agente: Letras de parcela que devolvio el agente.
        correctas: Letras que da la regla calculada con codigo comun.

    Returns:
        Diccionario con "faltan", "sobran" y "coincide".
    """
    faltan = []
    for letra in correctas:
        if False:  # COMPLETAR (1)
            faltan.append(letra)
    sobran = []
    for letra in del_agente:
        if False:  # COMPLETAR (2)
            sobran.append(letra)
    return {
        "faltan": sorted(faltan),
        "sobran": sorted(sobran),
        "coincide": len(faltan) == 0 and len(sobran) == 0,
    }


prueba = comparar_con_la_regla(["A", "B"], ["B", "C"])
utils.comprobar(
    prueba["faltan"] == ["C"] and prueba["sobran"] == ["A"],
    "revisa (1) y (2): faltan tiene que ser ['C'] y sobran ['A']",
)
utils.comprobar(
    comparar_con_la_regla(["A"], ["A"])["coincide"],
    "dos listas iguales tienen que dar coincide = True",
)

Todavia no: revisa (1) y (2): faltan tiene que ser ['C'] y sobran ['A']
Correcto.


A partir de acá usamos `utils.comparar_con_la_regla`, que hace lo mismo que la tuya.

In [7]:
filas_de_verificacion = []
for nombre_del_pedido, respuestas in resultados.items():
    for posicion in range(len(respuestas)):
        respuesta = respuestas[posicion]
        numero = posicion + 1
        comparacion = utils.comparar_con_la_regla(
            parcelas_de(respuesta), alertas_correctas
        )
        filas_de_verificacion.append({
            "pedido": nombre_del_pedido,
            "corrida": numero,
            "faltan": comparacion["faltan"],
            "sobran": comparacion["sobran"],
            "coincide con la regla": comparacion["coincide"],
        })

utils.tabla(filas_de_verificacion)

,pedido,corrida,faltan,sobran,coincide con la regla
0,ambiguo,1,"[B, D, E, F, G]",[],False
1,ambiguo,2,"[B, D, E, F, G]",[],False
2,ambiguo,3,"[B, D, E, F, G]",[],False
3,con criterios,1,"[B, D, E, F, G]",[],False
4,con criterios,2,[],[],True
5,con criterios,3,[],[],True


> **Para mirar.**
>
> - En las tres corridas ambiguas faltan las cinco parcelas, pero esa cifra mezcla dos cosas: que el modelo no entregó una lista y que no coincidió con la regla. La tabla no las separa.
> - `sobran` vacío no significa prudencia: una lista vacía no puede tener letras de más.
> - Las dos corridas con criterios que devolvieron un diccionario coinciden con la regla: sin diferencias en ninguna dirección.

In [8]:
informe_de_ejemplo = ""
respuesta_de_ejemplo = {}
for respuesta in resultados["con criterios"]:
    if isinstance(respuesta, dict) and "informe" in respuesta:
        informe_de_ejemplo = str(respuesta["informe"])
        respuesta_de_ejemplo = respuesta
        break
print("Informe de una corrida con criterios:")
print(informe_de_ejemplo)

Informe de una corrida con criterios:
Las parcelas ['B', 'D', 'E', 'F', 'G'] están en alerta porque al menos una de las condiciones se cumple: temperatura por encima de 34 °C, humedad por debajo de 40 %, lluvia nula o el estado de cultivo es distinto de 'normal'.


> **Para mirar.**
>
> - La lista de este informe coincide con la regla, y por eso pasa la verificación. El texto, en cambio, dice «por encima de 34 °C» y «por debajo de 40 %», mientras que la regla incluye el 34 y el 40.
> - La comparación revisa las letras, no la redacción: el informe lo tiene que leer una persona.

## 2. Una compuerta antes de guardar

El informe va a una carpeta que lee el técnico. Antes de guardarlo ponemos una **compuerta** con dos condiciones: que una persona lo haya **aprobado** y que las parcelas **coincidan con la regla**. La aprobación es una decisión de una persona, no del programa: la ejecución del cuaderno no puede aprobar nada sola.

### Microlaboratorio 2 — La compuerta

**Tarea.** Completar las dos condiciones que impiden guardar un informe.  
**Producto.** `guardar_si_se_aprueba`, comprobada con tres casos.  
**Antes de empezar.** Sección 1.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) primer `if` | verdadero cuando la decisión **no** es `"aprobar"` | comparar `decision` con `"aprobar"` usando `!=` |
| (2) segundo `if` | verdadero cuando las parcelas **no** coinciden con la regla | `not comparacion["coincide"]` |

**Comprobación.** Con la decisión `"pendiente"` no se guarda; con `"aprobar"` y parcelas correctas se guarda; con `"aprobar"` y parcelas equivocadas no se guarda.

> **Ayuda 1.** Las dos condiciones comienzan con un `return`: si se cumple una, la función termina sin escribir el archivo.  
> **Ayuda 2.** El orden importa para el mensaje: primero se pide la aprobación y después se verifica el contenido.

In [9]:
def guardar_si_se_aprueba(
    informe: str,
    del_agente: list[str],
    correctas: list[str],
    decision: str,
    nombre: str,
) -> str:
    """Guarda un informe solo si una persona lo aprobo y coincide.

    Args:
        informe: Texto del informe.
        del_agente: Letras de parcela que dijo el agente.
        correctas: Letras que da la regla calculada con codigo comun.
        decision: "aprobar" o cualquier otra palabra.
        nombre: Nombre del archivo, dentro de la carpeta salida.

    Returns:
        Un mensaje que dice si se guardo y por que.
    """
    comparacion = utils.comparar_con_la_regla(del_agente, correctas)
    if False:  # COMPLETAR (1)
        return "no se guardo: falta la aprobacion de una persona"
    if False:  # COMPLETAR (2)
        return "no se guardo: las parcelas no coinciden con la regla"
    carpeta = Path("salida")
    carpeta.mkdir(exist_ok=True)
    (carpeta / nombre).write_text(informe, encoding="utf-8")
    return f"guardado en salida/{nombre}"


mensaje_pendiente = guardar_si_se_aprueba(
    "informe", alertas_correctas, alertas_correctas, "pendiente",
    "prueba_1.txt",
)
mensaje_equivocado = guardar_si_se_aprueba(
    "informe", ["A"], alertas_correctas, "aprobar", "prueba_2.txt"
)
mensaje_correcto = guardar_si_se_aprueba(
    "informe", alertas_correctas, alertas_correctas, "aprobar", "prueba_3.txt"
)
utils.comprobar(
    mensaje_pendiente.startswith("no se guardo: falta"),
    "revisa (1): con la decision pendiente no se tiene que guardar",
)
utils.comprobar(
    mensaje_equivocado.startswith("no se guardo: las parcelas"),
    "revisa (2): con parcelas equivocadas no se tiene que guardar",
)
utils.comprobar(
    mensaje_correcto.startswith("guardado"),
    "con aprobacion y parcelas correctas se tiene que guardar",
)

Todavia no: revisa (1): con la decision pendiente no se tiene que guardar
Todavia no: revisa (2): con parcelas equivocadas no se tiene que guardar
Correcto.


La compuerta que usamos a partir de acá es `utils.guardar_si_se_aprueba`, con el mismo comportamiento. Antes de ejecutar la celda siguiente, **leé el informe de la sección anterior**: la variable `decision_humana` empieza en `"pendiente"` y solo vos podés cambiarla a `"aprobar"`.

In [10]:
# Cambiala a "aprobar" solo despues de leer el informe.
decision_humana = "pendiente"

informe_a_guardar = informe_de_ejemplo
parcelas_a_guardar = parcelas_de(respuesta_de_ejemplo)

mensaje = utils.guardar_si_se_aprueba(
    informe_a_guardar,
    parcelas_a_guardar,
    alertas_correctas,
    decision_humana,
    "informe_aprobado.txt",
)
print(mensaje)

no se guardo: falta la aprobacion de una persona


> **Para mirar.**
>
> - Mientras `decision_humana` diga `"pendiente"`, el informe no se guarda, por correcto que sea: ejecutar el cuaderno entero no aprueba nada.
> - La compuerta comprueba que las letras coincidan con la regla; no comprueba que la regla sea buena ni que la redacción sea fiel. Eso sigue en manos de quien aprueba.

## 3. Los mismos pedidos con Gemini

Corremos los dos pedidos con Gemini. Es opcional y hace seis corridas (tres por pedido), con una pausa entre una y otra por los límites del plan gratuito.

In [11]:
if utils.hay_clave("GEMINI_API_KEY"):
    modelo_remoto = utils.crear_modelo_gemini()
    filas_remotas = []
    for nombre_del_pedido, texto_del_pedido in pedidos.items():
        for corrida in range(1, 4):
            agente_remoto = CodeAgent(
                tools=[leer_parcelas], model=modelo_remoto,
                max_steps=5, verbosity_level=-1,
            )
            try:
                respuesta_remota = agente_remoto.run(texto_del_pedido)
            except Exception as error:
                respuesta_remota = f"Error: {error}"
            comparacion = utils.comparar_con_la_regla(
                parcelas_de(respuesta_remota), alertas_correctas
            )
            filas_remotas.append({
                "pedido": nombre_del_pedido,
                "corrida": corrida,
                "coincide con la regla": comparacion["coincide"],
            })
            time.sleep(6)
    tabla_remota = utils.tabla(filas_remotas)
else:
    tabla_remota = None
    utils.avisar_sin_clave("GEMINI_API_KEY")

tabla_remota

Sin GEMINI_API_KEY: se omite esta parte. Copia .env.example como .env, completa la clave y reinicia el kernel.


> **Para mirar.** Con clave, compará con la verificación de la sección 1. Un modelo más grande puede interpretar mejor el pedido ambiguo, pero sigue sin decir con qué criterio decidió: la verificación contra la regla es la que permite confiar o no. Sin clave, esta parte queda sin ejecutar.

## Qué te llevás

- Un pedido ambiguo produce informes que **no se pueden auditar**; con criterios escritos, el resultado se puede **comparar contra una regla calculada con código común** y el desacuerdo se mide (`faltan` y `sobran`).
- Que el informe sea convincente no lo hace correcto: la verificación tiene que ser **automática y anterior** a que una persona lo lea.
- La aprobación humana es una **compuerta**: nada se guarda ni se envía sin ella, y el programa no puede aprobar por su cuenta. Qué se verifica y quién firma son decisiones de diseño, no del modelo.

**Sigue:** una práctica integradora donde el agente combina dos herramientas (una búsqueda en documentos y un cálculo) y se compara con el flujo fijo de la unidad anterior (cuaderno 04).